# Capítulo 8 · Generación e inferencia

Cómo elegir cada carácter (voraz, temperatura, top-k y top-p), cómo medir la calidad de lo generado, la caché de claves y valores y la memoria. Usa el GPT entrenado en el capítulo 7, guardado en `material/gpt_quijote.pt`; no entrena nada. PyTorch en CPU. El cuaderno completo tarda un par de minutos.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo

Las clases son las de los capítulos 6 y 7. En Colab, la celda descarga el modelo y la novela si no están.

In [1]:
"""Capítulo 8: generación e inferencia con el GPT del Quijote. CPU."""
import math
import re
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

RAIZ = Path(".")
MODELO = RAIZ / "material" / "gpt_quijote.pt"
TEXTO = RAIZ / "material" / "quijote.txt"
for archivo in (MODELO, TEXTO):  # en Colab, los descargamos del repositorio
    if not archivo.exists():
        import urllib.request
        archivo.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/" + archivo.name, archivo)
INICIO = "En un lugar de la Mancha"


# --- El modelo de los capítulos 6 y 7 ------------------------------------

class Cabeza(nn.Module):
    def __init__(self, config):
        super().__init__()
        C, T = config["dimension"], config["contexto"]
        tamano = C // config["cabezas"]
        self.consulta = nn.Linear(C, tamano, bias=False)
        self.clave = nn.Linear(C, tamano, bias=False)
        self.valor = nn.Linear(C, tamano, bias=False)
        self.register_buffer("mascara", torch.tril(torch.ones(T, T)))

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.consulta(x), self.clave(x), self.valor(x)
        puntuaciones = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        futuro = self.mascara[:T, :T] == 0
        puntuaciones = puntuaciones.masked_fill(futuro, float("-inf"))
        return F.softmax(puntuaciones, dim=-1) @ v


class Bloque(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config["dimension"]
        self.cabezas = nn.ModuleList(
            [Cabeza(config) for _ in range(config["cabezas"])])
        self.proyeccion = nn.Linear(C, C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.ReLU(),
                                 nn.Linear(4 * C, C))
        self.norma1, self.norma2 = nn.LayerNorm(C), nn.LayerNorm(C)

    def forward(self, x):
        h = self.norma1(x)
        atencion = torch.cat([cabeza(h) for cabeza in self.cabezas], dim=-1)
        x = x + self.proyeccion(atencion)
        return x + self.mlp(self.norma2(x))


class GPT(nn.Module):
    def __init__(self, vocabulario, config):
        super().__init__()
        C = config["dimension"]
        self.config = config
        self.embedding_simbolos = nn.Embedding(vocabulario, C)
        self.embedding_posiciones = nn.Embedding(config["contexto"], C)
        self.bloques = nn.Sequential(
            *[Bloque(config) for _ in range(config["capas"])])
        self.norma = nn.LayerNorm(C)
        self.salida = nn.Linear(C, vocabulario)

    def forward(self, ids):
        posiciones = torch.arange(ids.shape[1])
        x = self.embedding_simbolos(ids) + self.embedding_posiciones(posiciones)
        return self.salida(self.norma(self.bloques(x)))


def cargar_modelo(ruta=MODELO):
    guardado = torch.load(ruta)
    simbolos = guardado["simbolos"]
    modelo = GPT(len(simbolos), guardado["config"])
    modelo.load_state_dict(guardado["modelo"])
    modelo.eval()
    return modelo, simbolos, {c: i for i, c in enumerate(simbolos)}

In [2]:
torch.set_num_threads(4)
modelo, simbolos, a_id = cargar_modelo()
print("Símbolos:", len(simbolos), "· parámetros:", sum(p.numel() for p in modelo.parameters()))

Símbolos: 92 · parámetros: 215260


## 2. La temperatura a mano

Los logits del ejercicio 1, divididos entre tres temperaturas distintas.

In [3]:
logits = torch.tensor([3.0, 1.0, 0.0])
for temperatura in (1.0, 0.5, 2.0):
    print(temperatura, F.softmax(logits / temperatura, dim=-1).round(decimals=3))

1.0 tensor([0.8440, 0.1140, 0.0420])
0.5 tensor([0.9800, 0.0180, 0.0020])
2.0 tensor([0.6290, 0.2310, 0.1400])


## 3. Elegir y generar

In [4]:
# --- Cómo elegir el siguiente carácter --------------------------------------

def elegir(logits, generador, temperatura=1.0, k=None, p=None):
    """Elige un identificador a partir de los logits de una posición."""
    if temperatura == 0:
        return int(torch.argmax(logits))  # voraz: siempre el más probable
    logits = logits / temperatura
    if k is not None:  # top-k: solo los k más probables
        umbral = torch.topk(logits, k).values[-1]
        logits = logits.masked_fill(logits < umbral, float("-inf"))
    probabilidades = F.softmax(logits, dim=-1)
    if p is not None:  # top-p: los más probables hasta sumar p
        orden = torch.argsort(probabilidades, descending=True)
        ordenadas = probabilidades[orden]
        antes = torch.cumsum(ordenadas, dim=0) - ordenadas  # suma previa
        probabilidades[orden[antes >= p]] = 0
        probabilidades = probabilidades / probabilidades.sum()
    return int(torch.multinomial(probabilidades, 1, generator=generador))


@torch.no_grad()
def generar(modelo, simbolos, a_id, inicio, n, semilla=7, **estrategia):
    generador = torch.Generator().manual_seed(semilla)
    contexto = modelo.config["contexto"]
    ids = [a_id[c] for c in inicio]
    for _ in range(n):
        logits = modelo(torch.tensor([ids[-contexto:]]))[0, -1]
        ids.append(elegir(logits, generador, **estrategia))
    return "".join(simbolos[i] for i in ids)

In [5]:
for estrategia in (dict(temperatura=0), dict(temperatura=0.8), dict(temperatura=1.5),
                   dict(temperatura=1.0, k=5), dict(temperatura=1.0, p=0.9)):
    print(estrategia)
    print("   ", repr(generar(modelo, simbolos, a_id, INICIO, 150, **estrategia)))

{'temperatura': 0}
    'En un lugar de la Mancha, y que le dijo que el caballero de la mano de la\nmano de la caballería de la cabeza de la caballería de la caballería de la\ncaballería de la caballer'
{'temperatura': 0.8}
    'En un lugar de la Mancha y los astoros\n\nEsas cautivos traían nos hacía hacer y de las llanos de sosos trocas a la\nsaben. ¡Piense. Yo es más determinacimiente de su esperada c'
{'temperatura': 1.5}
    'En un lugar de la Mancha L6onera\nGolSedó hastcao; y qque requiera las cosas,\ny jamás, llanden, qué\nviote.\n\n— ¡ólo sabre quíz se ejetiendrán —replixóCasmiena) dejáoisto\nolicon'
{'temperatura': 1.0, 'k': 5}
    'En un lugar de la Mancha y desdaba de llevar cartado, pues,\nconfalles donde cosas legunas, y es más comenzarse la desenero, y caballero\nque aquí por la desate de lastar libra'
{'temperatura': 1.0, 'p': 0.9}
    'En un lugar de la Mancha se ha aguado de salen otra\nser ventando y cuando serás alto, llaman de servido que en la sazón que\nles esponde

## 4. Medir la calidad

Fracción de palabras reales y variedad de parejas de palabras en 2000 caracteres por estrategia. Tarda alrededor de un minuto y medio; los tiempos dependen del equipo.

In [6]:
# --- Medir la calidad de un texto generado ----------------------------------

def palabras(texto):
    return re.findall(r"[a-záéíóúüñ]+", texto.lower())


def calidad(texto, palabras_quijote):
    lista = palabras(texto)
    reales = sum(palabra in palabras_quijote for palabra in lista) / len(lista)
    parejas = list(zip(lista, lista[1:]))
    variedad = len(set(parejas)) / len(parejas)
    return reales, variedad

In [7]:
palabras_quijote = set(palabras(TEXTO.read_text(encoding="utf-8")))
estrategias = {
    "voraz": dict(temperatura=0),
    "temperatura 0,5": dict(temperatura=0.5),
    "temperatura 0,8": dict(temperatura=0.8),
    "temperatura 1,0": dict(temperatura=1.0),
    "temperatura 1,5": dict(temperatura=1.5),
    "top-k 5": dict(temperatura=1.0, k=5),
    "top-p 0,9": dict(temperatura=1.0, p=0.9),
}
tabla = {}
for nombre, estrategia in estrategias.items():
    comienzo = time.perf_counter()
    texto = generar(modelo, simbolos, a_id, INICIO, 2000, **estrategia)
    ms = 1000 * (time.perf_counter() - comienzo) / 2000
    reales, variedad = calidad(texto[len(INICIO):], palabras_quijote)
    tabla[nombre] = (reales, variedad, ms)
    print("%-16s palabras reales %.3f · variedad %.3f · %.1f ms/car." % (
        nombre, reales, variedad, ms))
    print("   ", repr(texto[:200]))

voraz            palabras reales 1.000 · variedad 0.042 · 6.0 ms/car.
    'En un lugar de la Mancha, y que le dijo que el caballero de la mano de la\nmano de la caballería de la cabeza de la caballería de la caballería de la\ncaballería de la caballería de la caballería de la '
temperatura 0,5  palabras reales 0.944 · variedad 0.759 · 7.2 ms/car.
    'En un lugar de la Mancha y de la conoce de la buena que le que se le\ndio de la mano del alma de sus señoras, que los como si no es tenerme en el\npalabra en el mundo a los dos que se los míos de mi mes'
temperatura 0,8  palabras reales 0.838 · variedad 0.934 · 6.1 ms/car.
    'En un lugar de la Mancha y los astoros\n\nEsas cautivos traían nos hacía hacer y de las llanos de sosos trocas a la\nsaben. ¡Piense. Yo es más determinacimiente de su esperada como tan\njás mí, en tengo s'
temperatura 1,0  palabras reales 0.780 · variedad 0.963 · 6.1 ms/car.
    'En un lugar de la Mancha se nuestro Sancho Fenerme Bamante,\nconfalles con sucedia leg

## 5. La caché de claves y valores

In [8]:
# --- Una caché de claves y valores ------------------------------------------

@torch.no_grad()
def paso_con_cache(modelo, identificador, posicion, cache):
    """Calcula los logits de una posición nueva reutilizando el pasado."""
    x = (modelo.embedding_simbolos.weight[identificador]
         + modelo.embedding_posiciones.weight[posicion]).view(1, -1)
    for b, bloque in enumerate(modelo.bloques):
        h = bloque.norma1(x)
        salidas = []
        for c, cabeza in enumerate(bloque.cabezas):
            claves, valores = cache.setdefault((b, c), ([], []))
            claves.append(cabeza.clave(h))
            valores.append(cabeza.valor(h))
            K, V = torch.cat(claves), torch.cat(valores)
            q = cabeza.consulta(h)
            pesos = F.softmax(q @ K.T / math.sqrt(K.shape[-1]), dim=-1)
            salidas.append(pesos @ V)
        x = x + bloque.proyeccion(torch.cat(salidas, dim=-1))
        x = x + bloque.mlp(bloque.norma2(x))
    return modelo.salida(modelo.norma(x))[0]


@torch.no_grad()
def generar_con_cache(modelo, simbolos, a_id, inicio, n):
    """Generación voraz con caché; el texto completo cabe en el contexto."""
    assert len(inicio) + n <= modelo.config["contexto"]
    cache, ids = {}, [a_id[c] for c in inicio]
    for posicion, identificador in enumerate(ids):
        logits = paso_con_cache(modelo, identificador, posicion, cache)
    for _ in range(n):
        ids.append(int(torch.argmax(logits)))
        logits = paso_con_cache(modelo, ids[-1], len(ids) - 1, cache)
    return "".join(simbolos[i] for i in ids)


@torch.no_grad()
def comprobar_cache(modelo, ids):
    """Diferencia máxima entre los logits con y sin caché."""
    completos = modelo(torch.tensor([ids]))[0]
    cache = {}
    paso_a_paso = torch.stack(
        [paso_con_cache(modelo, identificador, posicion, cache)
         for posicion, identificador in enumerate(ids)])
    return float((completos - paso_a_paso).abs().max())


def cronometrar(funcion, repeticiones=5):
    comienzo = time.perf_counter()
    for _ in range(repeticiones):
        resultado = funcion()
    return resultado, (time.perf_counter() - comienzo) / repeticiones

In [9]:
ids = [a_id[c] for c in INICIO + ", de cuyo nombre no quiero acordarme"]
print("Diferencia máxima con y sin caché:", comprobar_cache(modelo, ids))
sin_cache = lambda: generar(modelo, simbolos, a_id, "E", 63, temperatura=0)
con_cache = lambda: generar_con_cache(modelo, simbolos, a_id, "E", 63)
texto_sin, tiempo_sin = cronometrar(sin_cache)
texto_con, tiempo_con = cronometrar(con_cache)
print("Mismo texto con y sin caché:", texto_sin == texto_con)
print("Sin caché: %.1f ms · con caché: %.1f ms" % (1000 * tiempo_sin,
                                                   1000 * tiempo_con))

Diferencia máxima con y sin caché: 5.9604644775390625e-06
Mismo texto con y sin caché: True
Sin caché: 341.6 ms · con caché: 170.2 ms


## 6. Memoria

In [10]:
# --- Memoria --------------------------------------------------------------

def memoria_parametros(n_parametros, bytes_por_numero=4):
    return n_parametros * bytes_por_numero


def memoria_cache(bloques, dimension, contexto, bytes_por_numero=4):
    # dos tablas (claves y valores) por bloque, de contexto x dimensión
    return 2 * bloques * dimension * contexto * bytes_por_numero

In [11]:
n = sum(parametro.numel() for parametro in modelo.parameters())
print("Parámetros:", n, "· memoria:", memoria_parametros(n), "bytes")
print("Caché con 64 posiciones:", memoria_cache(4, 64, 64), "bytes")

Parámetros: 215260 · memoria: 861040 bytes
Caché con 64 posiciones: 131072 bytes


## Ejercicios

1. Calcula, a mano o con una calculadora, softmax de los logits [3, 1, 0] con temperaturas 1, 0,5 y 2. ¿Qué le pasa a la probabilidad de la primera salida?

2. ¿Qué temperatura hace que los logits [3, 1, 0] den un reparto casi uniforme? ¿Y casi voraz? Razónalo sin calcular.

3. Con los logits [3, 1, 0] y T=1, aplica top-k con k=2 y top-p con p=0,9. ¿Cuántas salidas conserva cada uno?

4. Sigue a mano el código de top-p de `elegir` con las probabilidades [0,1; 0,6; 0,3] y p=0,5: ¿qué valen `orden`, `ordenadas` y `antes`? ¿Qué caracteres pasan a cero?

5. Explica por qué la elección voraz entra en bucles con nuestro modelo y por qué el sorteo los evita.

6. Según la tabla del capítulo, ¿qué estrategia elegirías para generar un relato? ¿Y para completar automáticamente una palabra mientras alguien escribe? Justifica tu respuesta con las dos medidas.

7. Añade dos entradas al diccionario `estrategias` de `ejecutar`, con temperaturas 0,3 y 1,2, ejecuta el programa y añade sus filas a la tabla. ¿Siguen la misma tendencia?

8. Repite las filas de temperatura 1, temperatura 0,8, top-k y top-p con las semillas 1 a 5 (el parámetro `semilla` de `generar`). ¿Cuánto cambian las cifras? ¿Se mantiene el orden entre estrategias?

9. La medida de palabras reales cuenta como correcta una palabra que exista en el Quijote, aunque esté mal usada. Pon un ejemplo de texto con un 100 % de palabras reales y ningún sentido. ¿Qué otra medida propondrías?

10. Explica con tus palabras por qué la caché KV no cambia el resultado. ¿Qué papel juega la máscara causal?

11. Llama a `generar_con_cache` con el comienzo `"E"` y n=64. ¿Por qué se detiene el programa? ¿Qué pasaría si quitáramos el `assert`?

12. ¿Cuántas posiciones procesa el modelo sin caché para generar 127 caracteres a partir de uno, si el contexto fuera de 128? ¿Y con caché?

13. Calcula la memoria de la caché de nuestro modelo si el contexto fuera de 1024 posiciones. ¿Y la de los parámetros si guardáramos cada número con 2 bytes?

14. Tabla de calidad, tiempo y memoria. Con los resultados de tu ejecución, completa una tabla con las siete estrategias, sus dos medidas de calidad y su tiempo por carácter, y añade la memoria de los parámetros y de la caché. Redacta cinco líneas con tus conclusiones.

## Referencias

La función de muestreo de [nanoGPT](https://github.com/karpathy/nanoGPT), de Andrej Karpathy, usa la temperatura y top-k como aquí. Top-p se propuso en *The Curious Case of Neural Text Degeneration* (Holtzman y colaboradores, 2020). Texto de *Don Quijote de la Mancha*, de dominio público. Código y medidas propios.